# Human saliva family abundance and clinical associations

Figure 4 and Supplementary Figure 3 source analyses. Includes Welch tests, the sex-stratified panel workflow, Pearson correlations and the separate Benjamini–Hochberg correction.

In [ ]:
from pathlib import Path

# Jupyter normally starts the kernel in the notebook directory.
here = Path.cwd()
repo_root = next(parent for parent in (here, *here.parents)
                 if (parent / "requirements.txt").is_file()
                 and (parent / "mouse").is_dir() and (parent / "human").is_dir())
data_dir = repo_root / "data"
output_dir = repo_root / "outputs" / 'human/05_saliva_family_abundance_and_correlations'
output_dir.mkdir(parents=True, exist_ok=True)
notebook_output = output_dir


## Load the workbook and rank families by group Z-score differences

In [ ]:
import pandas as pd

# Define the file path
path = data_dir / 'Human_RawData_for_PCA_SALIVA.xlsx'

# Read the Excel file into a DataFrame
df = pd.read_excel(path)
# Remove rows where all values are NaN
df = df.dropna(how='all')

df


In [ ]:
# Define label columns (metadata)
label_cols = [
    'SampleID', 'Age', 'Gender', 'Treatment', 'Disease_Duration',
    'Symptom_Duration', 'H_Y_Score', 'Collection'
]

# Define feature columns (everything else)
feature_cols = [col for col in df.columns if col not in label_cols]

# Create separate DataFrames
df_labels = df[label_cols]
df_features = df[feature_cols]


In [ ]:
# ============================================================
# PCA Visualization with Linear Boundary & Accuracy Annotation
# Save Top 10 Differential Features
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# -------------------------------
# Global style (font + scaling)
# -------------------------------
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 14,
    'axes.titlesize': 18,
    'axes.labelsize': 16,
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 13,
    'legend.title_fontsize': 14,
    'figure.titlesize': 20
})

plt.style.use("default")
plt.rcParams['pdf.fonttype'] = 42       # Use TrueType fonts for vector outputs
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'   # Keep text editable in SVGs

# -------------------------------
# Ensure treatment order
# -------------------------------
df_labels["Treatment"] = pd.Categorical(
    df_labels["Treatment"],
    categories=["Control", "Parkinson"],
    ordered=True
)

# -------------------------------
# Compute global Z-scores
# -------------------------------
scaler = StandardScaler()
df_z = pd.DataFrame(
    scaler.fit_transform(df_features),
    columns=df_features.columns,
    index=df_features.index
)

# -------------------------------
# Compute mean Z-score per treatment
# -------------------------------
df_mean_z = df_z.groupby(df_labels["Treatment"]).mean().T
df_mean_z["Difference"] = df_mean_z["Control"] - df_mean_z["Parkinson"]

# -------------------------------
# Filter features by absolute difference threshold
# -------------------------------
threshold = 0.4
selected_features = df_mean_z.loc[df_mean_z["Difference"].abs() > threshold].index

num_selected = len(selected_features)
num_total = df_features.shape[1]
print(f"Kept {num_selected} of {num_total} taxa ({num_selected / num_total:.1%}) with |ΔZ| > {threshold}")

# -------------------------------
# Identify Top 10 absolute differences
# -------------------------------
top10_features = (
    df_mean_z.reindex(selected_features)
    .loc[:, "Difference"]
    .abs()
    .sort_values(ascending=False)
    .head(10)
    .index
)

top10_features = list(top10_features)
print("\nTop 10 most discriminative features (by |ΔZ|):")
for i, f in enumerate(top10_features, 1):
    print(f"{i}. {f}")

# -------------------------------
# PCA on filtered features
# -------------------------------


## Screen candidate families with Welch’s t-test

In [ ]:
# ============================================================
# Top 10 Features (Taxa) – Bar + Scatter + Significance Stars
# + Store Significant Differences
# ============================================================

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ttest_ind

# --- Prepare data ---
df_plot = df_features[top10_features].copy()
df_plot["Treatment"] = df_labels["Treatment"].values

# --- Styling ---

palette = {"Control": "#42A5F5", "Parkinson": "#EF5350"}

# Helper to turn p-values into stars
def p_to_star(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

# Dictionary to store significant taxa results
sig_dict = {}

for feature in top10_features:
    # --- Compute t-test between groups ---
    ctrl = df_plot.loc[df_plot["Treatment"] == "Control", feature]
    park = df_plot.loc[df_plot["Treatment"] == "Parkinson", feature]
    tstat, pval = ttest_ind(ctrl, park, equal_var=False, nan_policy='omit')

    # Compute mean difference (PD - Control)
    diff = park.mean() - ctrl.mean()

    star = p_to_star(pval)

    # --- Store significance results ---
    if pval < 0.05:
        direction = "increased in PD" if diff > 0 else "decreased in PD"
        sig_dict[feature] = {
            "p_value": pval,
            "significance": star,
            "direction": direction
        }


## Prepare the Parkinson cohort and rank significant families

In [ ]:
# ============================================================
# ▪ Correlation of Significant Taxa with Clinical Variables (PD only)
#   + Heatmap Visualization  —  Added Gender Column
# ============================================================

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import patheffects
from scipy.stats import pearsonr

# ============================================================
# ▪ Assemble df_pd (Parkinson only, with Gender included)
# ============================================================

df_pd = df_features.copy()

# Attach required metadata columns
for col in ["SampleID", "Treatment", "Gender", "H_Y_Score", "Symptom_Duration", "Disease_Duration"]:
    if col in df_labels.columns:
        df_pd[col] = df_labels[col].values

# Subset to Parkinson participants
df_pd = df_pd[df_pd["Treatment"] == "Parkinson"].copy()

# ============================================================
# ▪ Clinical Variables
# ============================================================


In [ ]:
# === Identify Top 4 differential taxa (smallest p-values) ===
sig_df = (
    pd.DataFrame.from_dict(sig_dict, orient="index")
      .reset_index()
      .rename(columns={"index": "Taxon"})
      .sort_values("p_value")
)
top4_taxa = sig_df["Taxon"].head(4).tolist()
top10_taxa = sig_df["Taxon"].head(10).tolist()


## Plot overall and sex-stratified abundance panels

In [ ]:
if "Sex" not in df_plot.columns:
    if "Gender" in df_labels.columns:
        df_plot = df_plot.copy()
        df_plot["Sex"] = df_labels.loc[df_plot.index, "Gender"].str.lower()
    else:
        df_plot["Sex"] = "unknown"


In [ ]:
# ============================================================
# ▪ GraphPad‑style Bar Plot Function (final version)
# ============================================================

plt.style.use("default")

def graphpad_barplot(
    df,
    group_col,
    value_col,
    ylabel="",
    title=None,
    stats_text="",
    group_order=None,
    bar_width=0.25,
    gap=0.125,
    corner_prop=0.01,
    facecolor="white",
    edgecolor="black",
    figsize=(3.6, 5),
    font="Arial",
    ylim=None
):
    import matplotlib.pyplot as plt
    from matplotlib.patches import PathPatch
    from matplotlib.path import Path
    import numpy as np
    import pandas as pd
    import itertools

    # Determine group order / positions
    if group_order is None:
        groups = list(df[group_col].unique())
    else:
        groups = group_order
    x_positions = np.arange(len(groups)) * (bar_width + gap)

    means = [df[df[group_col] == g][value_col].mean() for g in groups]
    ses = [df[df[group_col] == g][value_col].sem() for g in groups]

    fig, ax = plt.subplots(figsize=figsize)

    # ------------------------------------------------------------
    # ▪ Bars (with beveled top)
    # ------------------------------------------------------------
    for x, mean in zip(x_positions, means):
        half = bar_width / 2
        verts = [
            (x - half, 0),
            (x - half, mean - corner_prop * mean),
            (x - half + corner_prop * bar_width, mean),
            (x + half - corner_prop * bar_width, mean),
            (x + half, mean - corner_prop * mean),
            (x + half, 0),
            (x - half, 0),
        ]
        patch = PathPatch(
            Path(verts, [Path.MOVETO] + [Path.LINETO]*5 + [Path.CLOSEPOLY]),
            facecolor=facecolor, edgecolor=edgecolor,
            lw=3, zorder=2
        )
        ax.add_patch(patch)

    # ------------------------------------------------------------
    # ▪ Error bars
    # ------------------------------------------------------------
    ax.errorbar(
        x_positions, means, yerr=ses,
        fmt="none", ecolor=edgecolor,
        elinewidth=1.2, capsize=10, capthick=1.6, zorder=3
    )

    # ------------------------------------------------------------
    # ▪ Scatter points
    #    • Females = solid black
    #    • Males   = white face + black edge
    # ------------------------------------------------------------
    if "Sex" in df.columns:
        for x, g in zip(x_positions, groups):
            sub = df[df[group_col] == g]
            for _, row in sub.iterrows():
                sex = str(row["Sex"]).lower()
                if sex.startswith("m"):
                    face = "black"
                    edge = "black"
                elif sex.startswith("f"):
                    face = "white"
                    edge = "black"
                else:
                    face = "white"
                    edge = "black"
                ax.scatter(
                    x + np.random.normal(0, 0.03),
                    row[value_col],
                    facecolor=face,
                    edgecolor=edge,
                    linewidth=1.8,
                    s=70,
                    zorder=4
                )
    else:
        for x, g in zip(x_positions, groups):
            y_vals = df[df[group_col] == g][value_col]
            ax.scatter(
                np.random.normal(x, 0.03, size=len(y_vals)),
                y_vals,
                facecolor="white",
                edgecolor="black",
                linewidth=1.5,
                s=60,
                zorder=4
            )

    # ------------------------------------------------------------
    # ▪ Significance bar + stars (if 2 groups)
    # ------------------------------------------------------------
    y_base_max = df[value_col].max()
    if len(groups) == 2 and stats_text:
        y_stats = y_base_max + 0.15 * y_base_max
        tick_len = 0.05 * y_base_max
        x0, x1 = x_positions[0], x_positions[1]
        dx = (x1 - x0) * 0.05
        path = Path([
            (x0, y_stats - tick_len), (x0, y_stats),
            (x0 + dx, y_stats + 0.01 * y_base_max),
            (x1 - dx, y_stats + 0.01 * y_base_max),
            (x1, y_stats), (x1, y_stats - tick_len)
        ])
        ax.add_patch(
            PathPatch(path, facecolor="none",
                      edgecolor=edgecolor, lw=3, zorder=5)
        )
        ax.text(
            np.mean(x_positions), y_stats * 1.03, stats_text,
            ha="center", va="bottom", fontsize=22, fontweight="bold", family=font
        )

    # ------------------------------------------------------------
    # ▪ Axis styling / formatting
    # ------------------------------------------------------------
    ax.set_xticks(x_positions)
    ax.set_xticklabels(groups, fontsize=14, fontweight="bold", family=font)

    # Y‑axis label (bold + larger)
    ax.set_ylabel(ylabel, fontsize=18, fontweight="bold", family=font)
    ylab = ax.yaxis.get_label()
    ylab.set_fontweight("bold")
    ylab.set_fontfamily(font)
    ylab.set_fontsize(18)

    # Thicker spines
    for spine in ["bottom", "left"]:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2.2)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Larger tick labels, y‑numbers bold
    ax.tick_params(axis="x", width=2, length=6, labelsize=16)
    ax.tick_params(axis="y", width=2, length=6, labelsize=16)
    for lbl in ax.get_yticklabels():
        lbl.set_fontweight("bold")
        lbl.set_fontfamily(font)

    # ------------------------------------------------------------
    # ▪ Force baseline at zero (remove bottom gap)
    # ------------------------------------------------------------
    y_max = df[value_col].max()
    y_top = y_max * 1.25 if y_max > 0 else 1
    ax.set_ylim(0, y_top)
    ax.spines["left"].set_bounds(0, y_top)
    ax.spines["left"].set_position(("outward", 0))

    if ylim is not None:
        ax.set_ylim(ylim)

    plt.tight_layout()

    # ------------------------------------------------------------
    # ▪ Center title across figure
    # ------------------------------------------------------------
    if title:
        fig.text(
            0.5, 1.02, title,
            ha="center", va="bottom",
            fontsize=16, fontweight="bold", family=font
        )

    plt.show()
    return fig, ax


In [ ]:
import os
import json
import matplotlib as mpl
import matplotlib.pyplot as plt

# ─────────────────────────────
# Matplotlib font / output setup
# ─────────────────────────────
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = 'none'
mpl.rcParams["font.family"]  = "Arial"

# ─────────────────────────────
# Output folder
# ─────────────────────────────
output_path = notebook_output / 'Human_Saliva_Top_Male_Female'
os.makedirs(output_path, exist_ok=True)

# ─────────────────────────────
# Loop over your top 10 taxa
# ─────────────────────────────
group_col = "Treatment"  # categorical grouping column in df_plot

# Dictionary to store y-axis limits
y_limits = {}

for taxon in top10_taxa:  # ← changed from top4_taxa to top10_taxa
    # Ensure df_plot already contains the “Sex” column!

    fig, ax = graphpad_barplot(
        df=df_plot,
        group_col=group_col,
        value_col=taxon,
        ylabel="Relative Abundance (A.U.)",
        title=f"{taxon}",
        stats_text=sig_dict.get(taxon, {}).get("significance", ""),
        facecolor="white",
        edgecolor="black",
        figsize=(3.6, 5),
        font="Arial"
    )

    # ─────────────────────────────
    # Store y-axis limits for later use
    # ─────────────────────────────
    y_limits[taxon] = ax.get_ylim()

    # ─­­ Optionally save each figure
    safe_name = taxon.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_path, f"{safe_name}_GraphPad.pdf")
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Saved → {save_path}")

# ─────────────────────────────
# Save y-axis limits for future plotting
# ─────────────────────────────
limits_path = os.path.join(output_path, "y_axis_limits.json")
with open(limits_path, "w") as f:
    json.dump(y_limits, f, indent=2)

print(f"Y-axis limits stored in: {limits_path}")


In [ ]:
import os
import json
from scipy.stats import ttest_ind
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ───────────────────────────────────────────────
# Matplotlib formatting (TrueType fonts / Arial)
# ───────────────────────────────────────────────
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = 'none'
mpl.rcParams["font.family"]  = "Arial"

# ───────────────────────────────────────────────
# Output folder for male‑only plots
# ───────────────────────────────────────────────
output_path_males = notebook_output / 'Human_Saliva_Top_MalesOnly'
os.makedirs(output_path_males, exist_ok=True)

# ───────────────────────────────────────────────
# Load the previously stored y-axis limits
# ───────────────────────────────────────────────
limits_path = notebook_output / 'Human_Saliva_Top_Male_Female' / 'y_axis_limits.json'

if os.path.exists(limits_path):
    with open(limits_path, "r") as f:
        y_limits = json.load(f)
else:
    print("⚠️ y_axis_limits.json not found — default limits will be used.")
    y_limits = {}

# ───────────────────────────────────────────────
# Loop over top 10 taxa — males only
# ───────────────────────────────────────────────
group_col = "Treatment"

for taxon in top10_taxa:
    # --- Filter to male samples only ---
    df_plot_male = df_plot[df_plot["Sex"].str.lower().str.startswith("m")].copy()

    # --- Re‑run t‑test using only male data ---
    groups = df_plot_male[group_col].unique()
    star = ""
    if len(groups) == 2:
        g1, g2 = groups
        t1 = df_plot_male[df_plot_male[group_col] == g1][taxon]
        t2 = df_plot_male[df_plot_male[group_col] == g2][taxon]
        if len(t1) >= 2 and len(t2) >= 2:
            _, pval = ttest_ind(t1, t2, equal_var=False, nan_policy="omit")
            star = p_to_star(pval)

    # --- Force all dots to be filled (pretend they are females) ---
    # df_plot_male_filled = df_plot_male.copy()
    # df_plot_male_filled["Sex"] = "F"   # ← forces filled black dots

    # --- Generate plot with stored y-limits if available ---
    fig, ax = graphpad_barplot(
        df=df_plot_male,
        group_col=group_col,
        value_col=taxon,
        ylabel="Relative Abundance (%)",
        title=f"{taxon} (Males only)",
        stats_text=star,
        facecolor="white",
        edgecolor="black",
        figsize=(3.6, 5),
        font="Arial",
        ylim=y_limits.get(taxon)  # ← pass y-limits directly here
    )

    # --- Save each figure ---
    safe_name = taxon.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_path_males, f"{safe_name}_MalesOnly.pdf")
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Saved → {save_path}")


In [ ]:
import os
import json
from scipy.stats import ttest_ind
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

# ───────────────────────────────────────────────
# Matplotlib formatting (TrueType fonts / Arial)
# ───────────────────────────────────────────────
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = 'none'
mpl.rcParams["font.family"]  = "Arial"

# ───────────────────────────────────────────────
# Output folder for female‑only plots
# ───────────────────────────────────────────────
output_path_females = notebook_output / 'Human_Saliva_Top_FemalesOnly'
os.makedirs(output_path_females, exist_ok=True)

# ───────────────────────────────────────────────
# Load the previously stored y-axis limits
# ───────────────────────────────────────────────
limits_path = notebook_output / 'Human_Saliva_Top_Male_Female' / 'y_axis_limits.json'

if os.path.exists(limits_path):
    with open(limits_path, "r") as f:
        y_limits = json.load(f)
else:
    print("⚠️ y_axis_limits.json not found — default limits will be used.")
    y_limits = {}

# ───────────────────────────────────────────────
# Loop over top 10 taxa — females only
# ───────────────────────────────────────────────
group_col = "Treatment"

for taxon in top10_taxa:
    # --- Filter to female samples only ---
    df_plot_female = df_plot[df_plot["Sex"].str.lower().str.startswith("f")].copy()

    # --- Re‑run t‑test using only female data ---
    groups = df_plot_female[group_col].unique()
    star = ""
    if len(groups) == 2:
        g1, g2 = groups
        t1 = df_plot_female[df_plot_female[group_col] == g1][taxon]
        t2 = df_plot_female[df_plot_female[group_col] == g2][taxon]
        if len(t1) >= 2 and len(t2) >= 2:
            _, pval = ttest_ind(t1, t2, equal_var=False, nan_policy="omit")
            star = p_to_star(pval)

    # --- Force all dots to be hollow (pretend they are males) ---
    # df_plot_female_hollow = df_plot_female.copy()
    # df_plot_female_hollow["Sex"] = "M"   # ← triggers hollow circles in graphpad_barplot

    # --- Generate plot with stored y-limits if available ---
    fig, ax = graphpad_barplot(
        df=df_plot_female,
        group_col=group_col,
        value_col=taxon,
        ylabel="Relative Abundance (%)",
        title=f"{taxon} (Females only)",
        stats_text=star,
        facecolor="white",
        edgecolor="black",
        figsize=(3.6, 5),
        font="Arial",
        ylim=y_limits.get(taxon)  # ← Apply consistent scale
    )

    # --- Save each figure ---
    safe_name = taxon.replace("/", "_").replace("\\", "_").replace(":", "_")
    save_path = os.path.join(output_path_females, f"{safe_name}_FemalesOnly.pdf")
    fig.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close(fig)

    print(f"Saved → {save_path}")


## Plot clinical correlations and calculate adjusted p-values

In [ ]:
# ============================================================
# Correlation Plots: Top 10 Taxa × Clinical Variables
# (Gender-based styling, regression line + orange CI, y≥0)
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.ticker import MaxNLocator

# ------------------------------------------------------------
# Matplotlib / font setup
# ------------------------------------------------------------
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"]  = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"]  = "Arial"
plt.style.use("default")

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial"],
    "pdf.fonttype": 42,     # Embed TrueType font, editable in Inkscape
    "ps.fonttype": 42,
    "svg.fonttype": "none", # Keep text as text, not paths
})

# ============================================================
# Identify Top 10 Differential Taxa
# ============================================================
sig_df = (
    pd.DataFrame.from_dict(sig_dict, orient="index")
      .reset_index()
      .rename(columns={"index": "Taxon"})
      .sort_values("p_value")
)
top10_taxa = sig_df["Taxon"].head(10).tolist()

# ============================================================
# Clinical Variables
# ============================================================
clinical_vars = ["H_Y_Score", "Disease_Duration", "Symptom_Duration"]
nice_labels = {
    "H_Y_Score": "H&Y Score",
    "Disease_Duration": "Disease Duration",
    "Symptom_Duration": "Symptom Duration",
}

# ============================================================
# Output path
# ============================================================
output_path = notebook_output / 'Human_Saliva_Top10_Correlations_AutoY_OrangeCI'
os.makedirs(output_path, exist_ok=True)

font = "Arial"

# ============================================================
# Helper: regression line + 95% CI (orange, low alpha)
# ============================================================
def plot_regression_with_ci(ax, x, y):
    """Plot regression line with 95% confidence interval (orange CI, clipped ≥0)."""
    import numpy as np
    from scipy import stats

    x = np.asarray(x)
    y = np.asarray(y)
    if len(x) < 2 or len(y) < 2:
        return

    slope, intercept, r_val, p_val, stderr = stats.linregress(x, y)
    x_pred = np.linspace(x.min(), x.max(), 100)
    y_pred = intercept + slope * x_pred

    # Residuals and 95% confidence interval
    y_fit = intercept + slope * x
    residuals = y - y_fit
    n = len(x)
    s_err = np.sqrt(np.sum(residuals**2) / (n - 2))
    t_val = stats.t.ppf(0.975, df=n - 2)
    mean_x = np.mean(x)
    confs = t_val * s_err * np.sqrt(1 / n + (x_pred - mean_x) ** 2 / np.sum((x - mean_x) ** 2))
    upper = y_pred + confs
    lower = np.maximum(0, y_pred - confs)  # clip CI at zero

    orange_ci = plt.cm.tab10(1)
    alpha = 0.25  # ← low transparency

    ax.fill_between(x_pred, lower, upper, color=orange_ci, alpha=alpha, zorder=1)
    ax.plot(x_pred, np.maximum(0, y_pred), color="black", linewidth=2.4, zorder=2)


# ============================================================
# Generate correlation plots
# ============================================================
for taxon in top10_taxa:
    for clinical in clinical_vars:
        x = df_pd[clinical]
        y = df_pd[taxon]
        mask = (~x.isna()) & (~y.isna())
        x, y = x[mask], y[mask]

        # ----------------------------------------------------
        # ▪ Determine Gender (males filled, females open)
        # ----------------------------------------------------
        if "Gender" in df_pd.columns:
            genders = df_pd.loc[mask, "Gender"].astype(str).str.strip().str.lower().str[0]
        else:
            genders = pd.Series([""] * len(x), index=x.index)

        # ----------------------------------------------------
        # ▪ Correlation statistics
        # ----------------------------------------------------
        if mask.sum() > 2:
            r_val, p_val = stats.pearsonr(x, y)
        else:
            r_val, p_val = np.nan, np.nan

        # ----------------------------------------------------
        # ▪ Figure setup
        # ----------------------------------------------------
        fig, ax = plt.subplots(figsize=(4, 4.5))

        # Regression line + low-alpha orange CI
        plot_regression_with_ci(ax, x, y)

        # Scatter points (male = filled, female = open)
        for xi, yi, gender in zip(x, y, genders):
            if gender == "m":
                ax.scatter(xi, yi, color="black", s=60, linewidth=1.6, alpha=0.9, zorder=3)
            elif gender == "f":
                ax.scatter(
                    xi, yi, facecolors="none", edgecolors="black",
                    s=60, linewidth=1.6, alpha=0.9, zorder=3
                )
            else:
                ax.scatter(
                    xi, yi, facecolors="white", edgecolors="black",
                    s=60, linewidth=1.6, alpha=0.9, zorder=3
                )

        # ----------------------------------------------------
        # ▪ Labels and r/p text
        # ----------------------------------------------------
        ax.set_xlabel(nice_labels.get(clinical, clinical),
                      fontsize=14, fontweight="bold", family=font)
        ax.set_ylabel(f"{taxon} Abundance",
                      fontsize=14, fontweight="bold", family=font)

        # Determine plot bounds
        x_min, x_max = float(x.min()), float(x.max())
        y_min, y_max = float(y.min()), float(y.max())
        x_range, y_range = x_max - x_min, max(y_max - y_min, 1e-9)

        # r/p annotation
        x_median = np.median(x)
        ha_pos = "left" if x_median > (x_min + 0.5 * x_range) else "right"
        text_x = x_min + 0.03 * x_range if ha_pos == "left" else x_max - 0.03 * x_range
        text_y = y_max - 0.05 * y_range
        ax.text(
            text_x, text_y,
            f"r = {r_val:.2f}\np = {p_val:.3f}",
            ha=ha_pos, va="top",
            fontsize=14, color="black", family=font,
            linespacing=1.2,
            bbox=dict(facecolor="white", alpha=0.0, edgecolor="none", pad=2)
        )

        # ----------------------------------------------------
        # ▪ GraphPad-style axes
        # ----------------------------------------------------
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_linewidth(1.8)
        ax.spines["left"].set_linewidth(1.8)
        ax.tick_params(width=1.8, length=5, labelsize=12)
        for lbl in ax.get_yticklabels() + ax.get_xticklabels():
            lbl.set_fontweight("bold")
            lbl.set_fontfamily(font)

        # ----------------------------------------------------
        # ▪ Auto y-limits (starting at zero)
        # ----------------------------------------------------
        ymin, ymax = ax.get_ylim()
        ax.set_ylim(0, ymax)  # keep top auto, bottom zero

        plt.tight_layout()
        fig.text(
            0.5, 1.02, f"{taxon} vs {nice_labels.get(clinical, clinical)}",
            ha="center", va="bottom", fontsize=14, fontweight="bold", family=font
        )

        # ----------------------------------------------------
        # ▪ Save or show
        # ----------------------------------------------------
        safe_taxon = taxon.replace("/", "_").replace("\\", "_").replace(":", "_")
        safe_clin = clinical.replace("/", "_").replace("\\", "_")
        save_path = os.path.join(output_path, f"{safe_taxon}_{safe_clin}_corr_AutoY_OrangeCI.pdf")

        fig.savefig(save_path, format="pdf", bbox_inches="tight")
        plt.show()
        print(f"Saved → {save_path}")


In [ ]:
# ============================================================
# Print FDR-corrected p-values for the clinical correlations
# ============================================================
from statsmodels.stats.multitest import multipletests

correlation_results = []

for taxon in top4_taxa:
    for clinical in clinical_vars:
        pairs = (
            df_pd[[clinical, taxon]]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        r_val, p_val = np.nan, np.nan

        if (
            len(pairs) >= 3
            and pairs[clinical].nunique() > 1
            and pairs[taxon].nunique() > 1
        ):
            r_val, p_val = pearsonr(pairs[clinical], pairs[taxon])

        correlation_results.append({
            "Taxon": taxon,
            "Clinical variable": nice_labels.get(clinical, clinical),
            "n": len(pairs),
            "r": r_val,
            "p_raw": p_val,
        })

df_corr_fdr = pd.DataFrame(
    correlation_results,
    columns=["Taxon", "Clinical variable", "n", "r", "p_raw"],
)
df_corr_fdr["p_FDR"] = np.nan

# Exclude undefined p-values from the correction
valid = np.isfinite(df_corr_fdr["p_raw"])

if valid.any():
    df_corr_fdr.loc[valid, "p_FDR"] = multipletests(
        df_corr_fdr.loc[valid, "p_raw"].to_numpy(),
        method="fdr_bh",
    )[1]

df_corr_fdr["Significant_FDR"] = df_corr_fdr["p_FDR"] < 0.05

print("\nClinical correlations — Benjamini–Hochberg FDR correction")
print(f"Number of valid tests corrected together: {valid.sum()}\n")
print(df_corr_fdr.to_string(
    index=False,
    float_format=lambda value: f"{value:.4g}",
    na_rep="NA",
))
